# Tools
* Models can request to call tools that perform tascs such as fetching data from a database, searching the web, or running code. Tools are pairings of:
<br>
    <t>1. A schema including the name of the tool, a decsiption, and/or argument definitions (often a JSON schema)
<br>
    <t>2. A function or coroutine to execute?

In [2]:
import os 
from langchain_openrouter import ChatOpenRouter

os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
model = ChatOpenRouter(model="openrouter/owl-alpha")
model

ChatOpenRouter(output_version=None, client=<openrouter.sdk.OpenRouter object at 0x7770989a3110>, openrouter_api_key=SecretStr('**********'), openrouter_api_base=None, app_url='https://docs.langchain.com', app_title='LangChain', model_name='openrouter/owl-alpha', model_kwargs={}, session_id=None)

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weatehr at a location""" ### Tools basically worcs on docstring
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])
### We can also use create agent as in "langintro.ipynb"

In [5]:
response = model_with_tools.invoke("Whats the weather in Boston")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(f"ID :{tool_call['id']}")
    print(f"Type: {tool_call['type']}")

content='' additional_kwargs={} response_metadata={'model_name': 'openrouter/owl-alpha', 'id': 'gen-1781199556-3RmuQCJ06Gvj7pzc5XtK', 'created': 1781199556, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter'} id='lc_run--019eb7c4-4b26-78a0-a54d-a0a7d77acb4e-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '216c7caa-89f6-4ec9-a3a4-3f7cc726d41d', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 230, 'output_tokens': 16, 'total_tokens': 246, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}
Tool: get_weather
Args: {'location': 'Boston'}
ID :216c7caa-89f6-4ec9-a3a4-3f7cc726d41d
Type: tool_call


### Tool Execution LOOP

In [6]:
# Step 1: Model generates the tool call
messages = [{"role":"user","content":"Whats the weather in boston"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results bac to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

# "THe currnt weatehr in bsoton is sunny"

It's sunny in Boston!


In [7]:
messages

[{'role': 'user', 'content': 'Whats the weather in boston'},
 AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openrouter/owl-alpha', 'id': 'gen-1781199958-cVRRCI87Bmf9rso8K6CJ', 'created': 1781199958, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter'}, id='lc_run--019eb7ca-720e-7603-a2c5-50f7ee5a2672-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '3ee7b958-a5b2-4dbe-a2a6-50cb24ff5f0d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 231, 'output_tokens': 16, 'total_tokens': 247, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}),
 ToolMessage(content="It's sunny in Boston", name='get_weather', tool_call_id='3ee7b958-a5b2-4dbe-a2a6-50cb24ff5f0d')]